In [1]:
# EfficientNet-B0 — Static PTQ (FX) with accuracy, size, runtime

import os, time, copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torchvision.models import efficientnet_b0
from sklearn.model_selection import train_test_split

from torch.ao.quantization import get_default_qconfig, QConfigMapping
from torch.ao.quantization.quantize_fx import prepare_fx, convert_fx

# Quant backend & device (CPU for PTQ eval) 
torch.backends.quantized.engine = "fbgemm"
device = torch.device("cpu")

# Dataset & transforms (match baseline) 
transform = transforms.Compose([
    transforms.Resize((224, 224), antialias=True),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2,
                           saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

data_dir = "DIR/horse/sapi279d-test_workspace/train"
dataset = datasets.ImageFolder(root=data_dir, transform=transform)
print(f"Total images: {len(dataset)}, Classes: {len(dataset.classes)}")

# Stratified split 85/10/5 
targets = np.array(dataset.targets)
train_idx, val_idx, test_idx = [], [], []
for c in np.unique(targets):
    idxs = np.where(targets == c)[0]
    tr, tmp = train_test_split(idxs, test_size=0.15, random_state=42, shuffle=True)
    va, te = train_test_split(tmp, test_size=1/3, random_state=42, shuffle=True)
    train_idx.extend(tr); val_idx.extend(va); test_idx.extend(te)

train_dataset, val_dataset, test_dataset = (
    Subset(dataset, train_idx),
    Subset(dataset, val_idx),
    Subset(dataset, test_idx)
)

batch_size = 128
num_workers = min(8, os.cpu_count() or 1)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          num_workers=num_workers, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                          num_workers=num_workers, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                          num_workers=num_workers, pin_memory=True)
print("DataLoaders ready")

# Load FP32 EfficientNet-B0 
ckpt = torch.load("efficientnet_b0_weights.pth", map_location="cpu")
ckpt = {k.replace("_orig_mod.", ""): v for k, v in ckpt.items()}

num_classes = len(dataset.classes)
model_fp32 = efficientnet_b0(weights=None)
model_fp32.classifier[1] = nn.Linear(model_fp32.classifier[1].in_features, num_classes)
missing, unexpected = model_fp32.load_state_dict(ckpt, strict=False)
print("Missing keys:", missing)
print("Unexpected keys:", unexpected)
model_fp32.eval().to(device)
print("FP32 EfficientNet-B0 loaded")

# FX graph mode prepare/convert 
qconfig = get_default_qconfig("fbgemm")
qconfig_mapping = QConfigMapping().set_global(qconfig)
example_inputs = (torch.randn(1, 3, 224, 224),)

prepared = prepare_fx(model_fp32, qconfig_mapping, example_inputs)

def calibrate_fx(model, loader, num_batches=50):
    model.eval()
    with torch.inference_mode():
        for i, (x, _) in enumerate(loader):
            if i >= num_batches: break
            model(x.to(device))

calibrate_fx(prepared, train_loader, num_batches=50)
print("Calibration done")

model_int8 = convert_fx(prepared).to(device).eval()
print("Static quantization complete")

# Eval / size / runtime 
def evaluate(model, loader):
    model.eval()
    corr, tot = 0, 0
    with torch.inference_mode():
        for x, y in loader:
            out = model(x.to(device))
            preds = out.argmax(1).cpu()
            corr += (preds == y).sum().item()
            tot  += y.size(0)
    return 100.0 * corr / tot

def get_model_size(m, path="__tmp__.pth"):
    torch.save(m.state_dict(), path)
    mb = os.path.getsize(path) / 1e6
    os.remove(path)
    return mb

def benchmark(m, loader, num_batches=50):
    m.eval()
    t0 = time.time()
    with torch.inference_mode():
        for i, (x, _) in enumerate(loader):
            if i >= num_batches: break
            _ = m(x.to(device))
    t1 = time.time()
    imgs = num_batches * loader.batch_size
    return (t1 - t0) / num_batches * 1000.0, (t1 - t0) / imgs * 1000.0

acc_fp32 = evaluate(model_fp32, test_loader)
acc_int8 = evaluate(model_int8,  test_loader)
print(f"FP32 Accuracy: {acc_fp32:.2f}%")
print(f"INT8 Accuracy (Static PTQ): {acc_int8:.2f}%")

size_fp32 = get_model_size(model_fp32)
size_int8 = get_model_size(model_int8)
print(f"FP32 size: {size_fp32:.2f} MB | INT8 size: {size_int8:.2f} MB")

b_fp32, i_fp32 = benchmark(model_fp32, test_loader)
b_int8, i_int8 = benchmark(model_int8,  test_loader)
print(f"FP32 Inference: {b_fp32:.2f} ms/batch | {i_fp32:.2f} ms/image")
print(f"INT8 Inference: {b_int8:.2f} ms/batch | {i_int8:.2f} ms/image")



Total images: 100000, Classes: 200
DataLoaders ready
Missing keys: []
Unexpected keys: []
FP32 EfficientNet-B0 loaded


/software/util/JupyterLab/alpha/share/pytorch_v2/lib/python3.10/site-packages/torch/ao/quantization/observer.py:220: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  warnings.warn(
/software/util/JupyterLab/alpha/share/pytorch_v2/lib/python3.10/site-packages/torch/ao/quantization/fx/utils.py:857: UserWarning: QConfig must specify a FixedQParamsObserver or a FixedQParamsFakeQuantize for fixed qparams ops, ignoring QConfig(activation=functools.partial(<class 'torch.ao.quantization.observer.HistogramObserver'>, reduce_range=True){'factory_kwargs': <function _add_module_to_qconfig_obs_ctr.<locals>.get_factory_kwargs_based_on_module_device at 0x149d66098550>}, weight=functools.partial(<class 'torch.ao.quantization.observer.PerChannelMinMaxObserver'>, dtype=torch.qint8, qscheme=torch.per_channel_symmetric){'factory_kwargs': <function _add_module_to_qconfig_obs_ctr.<locals>

Calibration done
Static quantization complete
FP32 Accuracy: 74.58%
INT8 Accuracy (Static PTQ): 34.14%
FP32 size: 17.35 MB | INT8 size: 5.03 MB
FP32 Inference: 1840.46 ms/batch | 14.38 ms/image
INT8 Inference: 1270.60 ms/batch | 9.93 ms/image
